In [1]:
import sys
sys.path.append('../..')
from sqlalchemy import create_engine, MetaData, select, Table
import schedule
import pandas as pd
from datetime import datetime
import time
from config import POSTGRES_UTEA
import math

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

metadata = MetaData()
reporte_tbl = Table("reporte", metadata, autoload_with=ENGINE, schema="datos_iag")
msj_whatsapp_tbl = Table("msj_whatsapp", metadata, autoload_with=ENGINE, schema="notificaciones_wsp")

In [2]:
def crear_mensaje(datos):    
    try:
        with ENGINE.begin() as conn:
            conn.execute(msj_whatsapp_tbl.insert(), datos.to_dict(orient='records'))
        print("✅ Se ha actualizado ")
    except Exception as e:
        print("❌ Error al insertar en tabla MSJ WHASTAPP", e)
    #return df

In [4]:
df = pd.read_excel(r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\ESTIMATIVAS\V02\links_planos_estimativas_mas_cod_ca.xlsx', sheet_name = 'contactos')

In [6]:
df

,cod_canero,nombre_canero,numero_cel
0,86,AGUILERA TARADELLES JOSE LUIS,67759316
1,86,AGUILERA TARADELLES JOSE LUIS,77010520
2,86,AGUILERA TARADELLES JOSE LUIS,68848905
3,180,ELISEO AMURRIO,67559958
4,388,AGROPECUARIA MARIANA SRL,77334096
...,...,...,...
177,42347,MERCADO KATERINE JUSTINIANO DE,76090266
178,42376,HURTADO ANTONIO PEREDO,70951751
179,42376,HURTADO ANTONIO PEREDO,78440305
180,42455,CIA IND. COMERCIAL HNOS. VICENTE S.R.L.,75525250


In [7]:
df['fecha_registro'] = datetime.now()
df['cod_canero'] = df['cod_canero']

df['mensaje'] = df.apply(lambda row: f"""🌱 *Planos de estimativas de producción* 🚜

Saludos estimado productor cañero: *{row['nombre_canero']}* 👋, le hacemos llegar los planos de estimativas de producción de las propiedades catastradas en UTEA. 📄📍

Estos planos son una herramienta más para determinar la producción total en la **campaña 2026** 📅. Pueden ser validados en campo ✅; si observa alguna anomalía, no dude en contactarnos. 📲

Quedamos atentos a sus consultas. 🙏✨""", axis=1)


df['enviado'] = False
df['fecha_enviado'] = None
df['numero_contac'] = '591' + df['numero_cel'].astype(str) + '@s.whatsapp.net'

In [24]:
df = df.drop(['COD_COS', 'NOMBRE', 'NUM', 'OBS'], axis=1)

In [8]:
df

,cod_canero,nombre_canero,numero_cel,fecha_registro,mensaje,enviado,fecha_enviado,numero_contac
0,86,AGUILERA TARADELLES JOSE LUIS,67759316,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59167759316@s.whatsapp.net
1,86,AGUILERA TARADELLES JOSE LUIS,77010520,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59177010520@s.whatsapp.net
2,86,AGUILERA TARADELLES JOSE LUIS,68848905,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59168848905@s.whatsapp.net
3,180,ELISEO AMURRIO,67559958,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59167559958@s.whatsapp.net
4,388,AGROPECUARIA MARIANA SRL,77334096,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59177334096@s.whatsapp.net
...,...,...,...,...,...,...,...,...
177,42347,MERCADO KATERINE JUSTINIANO DE,76090266,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59176090266@s.whatsapp.net
178,42376,HURTADO ANTONIO PEREDO,70951751,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59170951751@s.whatsapp.net
179,42376,HURTADO ANTONIO PEREDO,78440305,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59178440305@s.whatsapp.net
180,42455,CIA IND. COMERCIAL HNOS. VICENTE S.R.L.,75525250,2026-03-29 11:18:05.399189,🌱 *Planos de estimativas de producción* 🚜\n\nS...,False,None,59175525250@s.whatsapp.net


In [9]:
crear_mensaje(df)

✅ Se ha actualizado 
